# Warren Meeting Slides (May 2026)

Generates the slide deck for the Warren White meeting per Ann's email feedback.

**Slide order (from Ann's email):**
1. Intro: dataset and problem
2. Data and tools used
3. Fabs (1/Mm) vs EC (ug/m3) for four SPARTAN sites
4. Fabs (1/Mm) vs FTIR EC for four sites
5. Fabs (1/Mm) vs Aethalometer
6. Iron (Fe and Fe/EC ranges, with rough IMPROVE comparison)
7. Seasonality + checklist of what we've ruled out
8. Four SPARTAN sites overlaid on IMPROVE background, EC mass on filter (ug)
9. Four separate plots, one per site, 5-95 percentile shading on both axes
10. SPARTAN R/T plot per site (+ Warren's IMPROVE R/T plot for comparison) and Addis transmittance offset
11. Questions for Warren

**IMPORTANT:** IMPROVE FED data lives outside this repo (the Google Drive path Ahmad uses). Where IMPROVE data is required, this notebook checks for a local CSV/PKL and falls back to a placeholder so the notebook still runs end-to-end. Drop your IMPROVE FED extract at `IMPROVE_DATA_PATH` (set below) before running, or the IMPROVE layer on slides 8/9 will say `IMPROVE DATA NOT FOUND - PLACEHOLDER`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Repo paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent

SCRIPTS_DIR = REPO_ROOT / 'research' / 'ftir_hips_chem' / 'scripts'
for p in (SCRIPTS_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from config import SITES, MAC_VALUE  # noqa: E402
from data_matching import load_aethalometer_data, load_filter_data  # noqa: E402

OUTPUT_DIR = REPO_ROOT / 'research' / 'ftir_hips_chem' / 'output' / 'warren_meeting'
FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Drop IMPROVE FED extract here. Expected columns:
#   SiteCode, Date, EC_ugm3, Volume_m3, FABS_invMm, Fe_ugm3, R, T
# (Names below; rename when loading.)
IMPROVE_DATA_PATH = REPO_ROOT / 'research' / 'improve_hips_offset' / 'improve_fed_extract.pkl'

print(f'Repo: {REPO_ROOT}')
print(f'Output: {OUTPUT_DIR}')
print(f'IMPROVE expected at: {IMPROVE_DATA_PATH} (exists={IMPROVE_DATA_PATH.exists()})')


## Load SPARTAN data

For each of the four sites we build a per-filter dataframe with the columns we need on every slide:

- `FABS_invMm` - HIPS Fabs in inverse megameters (raw, **NOT** divided by MAC)
- `EC_ugm3` - FTIR EC concentration (ug/m^3)
- `EC_mass_ug` - EC mass on filter (ug). From `MassLoading_ug` in the unified filter dataset.
- `Fe_ugm3` - ChemSpec iron concentration
- `Aeth_ugm3` - Aethalometer IR BCc averaged over the filter day (ug/m^3)
- `Volume_m3`, `DepositArea_cm2`, `R`, `T`, `tau`, `t`, `r` - HIPS raw fields

In [ ]:
filter_data = load_filter_data()
aeth_data = load_aethalometer_data()

PARAMS_TO_PIVOT = [
    'EC_ftir', 'HIPS_Fabs', 'HIPS_T1', 'HIPS_R1',
    'HIPS_t', 'HIPS_r', 'HIPS_tau',
    'ChemSpec_Iron_PM2.5',
]

def build_site_table(site_name, site_code, df_aeth, filter_long):
    site_filters = filter_long[filter_long['Site'] == site_code].copy()
    if site_filters.empty:
        return None
    # Pivot Concentration by Parameter
    wide = (
        site_filters[site_filters['Parameter'].isin(PARAMS_TO_PIVOT)]
        .pivot_table(index=['FilterId', 'SampleDate', 'Volume_m3', 'DepositArea_cm2'],
                     columns='Parameter', values='Concentration', aggfunc='mean')
        .reset_index()
    )
    # Pull EC mass loading separately (concentration pivot drops MassLoading_ug)
    ec_mass = (
        site_filters[site_filters['Parameter'].eq('EC_ftir')][['FilterId', 'MassLoading_ug']]
        .drop_duplicates('FilterId')
        .rename(columns={'MassLoading_ug': 'EC_mass_ug'})
    )
    wide = wide.merge(ec_mass, on='FilterId', how='left')
    wide = wide.rename(columns={
        'EC_ftir': 'EC_ugm3',
        'HIPS_Fabs': 'FABS_invMm',
        'ChemSpec_Iron_PM2.5': 'Fe_ugm3',
        'HIPS_T1': 'T', 'HIPS_R1': 'R',
        'HIPS_t': 't', 'HIPS_r': 'r', 'HIPS_tau': 'tau',
    })
    # If EC mass missing, back-calculate from concentration*volume
    wide['EC_mass_ug'] = wide['EC_mass_ug'].fillna(wide['EC_ugm3'] * wide['Volume_m3'])
    # Aethalometer match by date (mean over +/- 1 day)
    if df_aeth is not None and 'IR BCc' in df_aeth.columns:
        aeth_idx = df_aeth.set_index(pd.to_datetime(df_aeth['day_9am']))['IR BCc'].sort_index()
        def lookup(d):
            d = pd.to_datetime(d)
            sl = aeth_idx.loc[d - pd.Timedelta(days=1):d + pd.Timedelta(days=1)]
            return np.nan if sl.empty else sl.mean() / 1000.0  # ng/m3 -> ug/m3
        wide['Aeth_ugm3'] = wide['SampleDate'].apply(lookup)
    else:
        wide['Aeth_ugm3'] = np.nan
    wide['site'] = site_name
    wide['site_color'] = SITES[site_name]['color']
    return wide

site_tables = {}
for site_name, cfg in SITES.items():
    df = build_site_table(site_name, cfg['code'], aeth_data.get(site_name), filter_data)
    if df is not None:
        site_tables[site_name] = df
        print(f"{site_name}: {len(df)} filters")


## IMPROVE data loader (placeholder-aware)

If the IMPROVE pickle/CSV is present at `IMPROVE_DATA_PATH`, this cell loads it and computes EC mass on filter (`EC_ugm3 * Volume_m3`). Otherwise we set `improve_df = None` and the IMPROVE-overlay slides display a 'PLACEHOLDER - LOAD IMPROVE DATA' watermark.

In [ ]:
def load_improve():
    if not IMPROVE_DATA_PATH.exists():
        return None
    if IMPROVE_DATA_PATH.suffix == '.pkl':
        df = pd.read_pickle(IMPROVE_DATA_PATH)
    else:
        df = pd.read_csv(IMPROVE_DATA_PATH)
    # Expected/standardized columns
    rename_map = {
        'EC_concentration_ugm3': 'EC_ugm3',
        'fAbs_invMm': 'FABS_invMm',
        'fAbs': 'FABS_invMm',
        'Fe': 'Fe_ugm3',
        'Iron_ugm3': 'Fe_ugm3',
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    # Limit to post-2003 per Warren's paper
    if 'Date' in df.columns:
        df = df[pd.to_datetime(df['Date']) >= '2003-01-01']
    if 'EC_mass_ug' not in df.columns and {'EC_ugm3', 'Volume_m3'}.issubset(df.columns):
        df['EC_mass_ug'] = df['EC_ugm3'] * df['Volume_m3']
    return df

improve_df = load_improve()
if improve_df is None:
    print('IMPROVE data not found - slides 8/9 will show placeholder watermark.')
else:
    print(f'IMPROVE rows loaded: {len(improve_df):,}')


In [ ]:
# Style helpers
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 15,
    'legend.fontsize': 13,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
})

SITE_ORDER = ['Addis_Ababa', 'Delhi', 'JPL', 'Beijing']
SITE_LABELS = {'Addis_Ababa': 'Addis Ababa', 'Delhi': 'Delhi', 'JPL': 'JPL/Pasadena', 'Beijing': 'Beijing'}
SITE_COLORS = {s: SITES[s]['color'] for s in SITE_ORDER}

def placeholder_watermark(ax, msg='IMPROVE DATA NOT FOUND\nPLACEHOLDER'):
    ax.text(0.5, 0.5, msg, transform=ax.transAxes, ha='center', va='center',
            color='red', fontsize=22, alpha=0.35, fontweight='bold')

def save_fig(fig, name):
    path = FIG_DIR / f'{name}.png'
    fig.savefig(path, dpi=180, bbox_inches='tight', facecolor='white')
    print(f'Saved: {path}')
    return path


## Slide 3 - Fabs (1/Mm) vs EC (ug/m3) for four sites

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None or df.empty:
        continue
    ax.scatter(df['EC_ugm3'], df['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=f"{SITE_LABELS[site]} (n={df['FABS_invMm'].notna().sum()})")
ax.set_xlabel('EC concentration (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('HIPS Fabs vs EC concentration - four SPARTAN sites')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
save_fig(fig, 'slide03_fabs_vs_ec_concentration')
plt.show()


## Slide 4 - Fabs (1/Mm) vs FTIR EC for four sites

Same comparison, FTIR EC explicitly. (If Slides 3 and 4 are intended to be the same, this is a duplicate kept per Ann's outline; otherwise edit this cell to swap in ChemSpec_EC for slide 3 above.)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None or df.empty:
        continue
    ax.scatter(df['EC_ugm3'], df['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=f"{SITE_LABELS[site]} (n={df['FABS_invMm'].notna().sum()})")
ax.set_xlabel('FTIR EC (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('HIPS Fabs vs FTIR EC - four SPARTAN sites')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
save_fig(fig, 'slide04_fabs_vs_ftir_ec')
plt.show()


## Slide 5 - Fabs (1/Mm) vs Aethalometer

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None or df.empty:
        continue
    sub = df.dropna(subset=['Aeth_ugm3', 'FABS_invMm'])
    if sub.empty:
        continue
    ax.scatter(sub['Aeth_ugm3'], sub['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=f"{SITE_LABELS[site]} (n={len(sub)})")
ax.set_xlabel('Aethalometer IR BCc (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('HIPS Fabs vs Aethalometer IR BCc')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
save_fig(fig, 'slide05_fabs_vs_aethalometer')
plt.show()


## Slide 6 - Iron (Fe and Fe/EC ranges)

Two panels:
- Left: Fe vs Fabs by site
- Right: Fe/EC ratio distributions by site

Add IMPROVE Fe and Fe/EC bands when IMPROVE data is loaded.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

ax = axes[0]
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None: continue
    sub = df.dropna(subset=['Fe_ugm3', 'FABS_invMm'])
    if sub.empty: continue
    ax.scatter(sub['Fe_ugm3'], sub['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=f'{SITE_LABELS[site]} (n={len(sub)})')
if improve_df is not None and 'Fe_ugm3' in improve_df.columns:
    fe_p5, fe_p95 = improve_df['Fe_ugm3'].quantile([0.05, 0.95])
    ax.axvspan(fe_p5, fe_p95, color='gray', alpha=0.18, label='IMPROVE Fe 5-95%')
elif improve_df is None:
    placeholder_watermark(ax, 'IMPROVE Fe band\nNOT LOADED')
ax.set_xlabel('Fe (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('Fe vs Fabs')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

ax = axes[1]
ratios, labels, colors = [], [], []
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None: continue
    r = (df['Fe_ugm3'] / df['EC_ugm3']).replace([np.inf, -np.inf], np.nan).dropna()
    if r.empty: continue
    ratios.append(r.values); labels.append(SITE_LABELS[site]); colors.append(SITE_COLORS[site])
if ratios:
    bp = ax.boxplot(ratios, labels=labels, patch_artist=True, showfliers=False)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.6)
if improve_df is not None and {'Fe_ugm3', 'EC_ugm3'}.issubset(improve_df.columns):
    imp_ratio = (improve_df['Fe_ugm3'] / improve_df['EC_ugm3']).replace([np.inf, -np.inf], np.nan).dropna()
    p5, p95 = imp_ratio.quantile([0.05, 0.95])
    ax.axhspan(p5, p95, color='gray', alpha=0.2, label='IMPROVE Fe/EC 5-95%')
    ax.legend(fontsize=11)
elif improve_df is None:
    placeholder_watermark(ax, 'IMPROVE Fe/EC band\nNOT LOADED')
ax.set_ylabel('Fe / EC')
ax.set_title('Fe/EC by site')
ax.grid(alpha=0.3, axis='y')
fig.suptitle('Iron - unlike pixelation, Fe should increase Fabs', fontsize=15)
fig.tight_layout()
save_fig(fig, 'slide06_iron')
plt.show()


## Slide 7 - Seasonality + checklist

Boxplots of Fabs by season, per site. Below, a short checklist of variables already ruled out.

In [ ]:
from src.config.multi_site_seasons import SITE_SEASONS  # noqa: E402

fig, axes = plt.subplots(1, 4, figsize=(20, 6), sharey=True)
for ax, site in zip(axes, SITE_ORDER):
    df = site_tables.get(site)
    if df is None or 'SampleDate' not in df.columns:
        ax.set_visible(False); continue
    df = df.copy()
    df['month'] = pd.to_datetime(df['SampleDate']).dt.month
    seasons = SITE_SEASONS.get(site, {})
    data, labels, colors = [], [], []
    for sname, info in seasons.items():
        vals = df[df['month'].isin(info['months'])]['FABS_invMm'].dropna().values
        if len(vals) == 0: continue
        data.append(vals); labels.append(sname.split(' (')[0]); colors.append(info['color'])
    if data:
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
        for patch, c in zip(bp['boxes'], colors):
            patch.set_facecolor(c); patch.set_alpha(0.65)
    ax.set_title(SITE_LABELS[site])
    ax.grid(alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=20)
axes[0].set_ylabel('HIPS Fabs (1/Mm)')
fig.suptitle('Seasonality of Fabs by site - Addis lacks the seasonal shape we see elsewhere', fontsize=15)
fig.tight_layout()
save_fig(fig, 'slide07_seasonality')
plt.show()

# Checklist of things tried/ruled out (rendered into the pptx as bullets)
RULED_OUT_CHECKLIST = [
    'Iron absorption (Fe / Fe-to-EC distribution)',
    'Seasonality - Addis shows no seasonal shape',
    'Smooth vs raw aethalometer (% difference)',
    'FTIR EC vs aethalometer agreement (no large offset)',
    'EC mass-loading range vs IMPROVE',
    'Filter sub-IDs / lot numbers as available',
    'HIPS MAC = 10 (not the source of the offset)',
    'Volume / flow ratio corrections (where applicable)',
]
print('\n'.join('- ' + x for x in RULED_OUT_CHECKLIST))


## Slide 8 - Four SPARTAN sites overlaid on IMPROVE background (EC mass on filter)

IMPROVE in gray (single-size dots, no shading); the four sites in distinct colors. This is the headline plot - we expect the SPARTAN scatter (especially Addis) to sit far above the IMPROVE cloud.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
if improve_df is not None and {'EC_mass_ug', 'FABS_invMm'}.issubset(improve_df.columns):
    sub = improve_df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])
    ax.scatter(sub['EC_mass_ug'], sub['FABS_invMm'], s=8, alpha=0.25,
               color='lightgray', label=f'IMPROVE (n={len(sub):,})', zorder=1)
else:
    placeholder_watermark(ax)
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None: continue
    sub = df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])
    if sub.empty: continue
    ax.scatter(sub['EC_mass_ug'], sub['FABS_invMm'], s=36, alpha=0.8,
               color=SITE_COLORS[site], edgecolor='black', linewidth=0.4,
               label=f'{SITE_LABELS[site]} (n={len(sub)})', zorder=3)
ax.set_xlabel('EC mass on filter (µg)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('Four SPARTAN sites vs IMPROVE - EC mass on filter')
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(alpha=0.3)
save_fig(fig, 'slide08_overlay_mass_on_filter')
plt.show()


## Slide 9 - Four separate plots, 5-95 percentile shading on both axes

One panel per SPARTAN site, IMPROVE in gray. Red/colored shading shows the 5-95 percentile of that site's EC-mass and Fabs simultaneously, to highlight where the bulk of the site's data lives versus the IMPROVE cloud.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
for ax, site in zip(axes, SITE_ORDER):
    if improve_df is not None and {'EC_mass_ug', 'FABS_invMm'}.issubset(improve_df.columns):
        sub = improve_df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])
        ax.scatter(sub['EC_mass_ug'], sub['FABS_invMm'], s=8, alpha=0.25,
                   color='lightgray', label='IMPROVE', zorder=1)
    else:
        placeholder_watermark(ax)
    df = site_tables.get(site)
    if df is None: continue
    sub = df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])
    if not sub.empty:
        x_lo, x_hi = sub['EC_mass_ug'].quantile([0.05, 0.95])
        y_lo, y_hi = sub['FABS_invMm'].quantile([0.05, 0.95])
        ax.axvspan(x_lo, x_hi, color=SITE_COLORS[site], alpha=0.10, zorder=2)
        ax.axhspan(y_lo, y_hi, color=SITE_COLORS[site], alpha=0.10, zorder=2)
        ax.scatter(sub['EC_mass_ug'], sub['FABS_invMm'], s=36, alpha=0.85,
                   color=SITE_COLORS[site], edgecolor='black', linewidth=0.4,
                   label=f'{SITE_LABELS[site]} (n={len(sub)})', zorder=3)
    ax.set_xlabel('EC mass on filter (µg)')
    ax.set_ylabel('HIPS Fabs (1/Mm)')
    ax.set_title(SITE_LABELS[site])
    ax.legend(loc='upper left', fontsize=11)
    ax.grid(alpha=0.3)
# Force consistent axis ranges across panels (ranges from union of data)
x_max = max([np.nanmax(d['EC_mass_ug'].values) for d in site_tables.values() if not d.empty] + [50])
y_max = max([np.nanmax(d['FABS_invMm'].values) for d in site_tables.values() if not d.empty] + [50])
for ax in axes:
    ax.set_xlim(0, x_max * 1.05)
    ax.set_ylim(0, y_max * 1.05)
fig.suptitle('Per-site view with 5-95 percentile shading on both axes', fontsize=15)
fig.tight_layout()
save_fig(fig, 'slide09_per_site_shaded')
plt.show()


## Slide 10 - SPARTAN R vs T per site (+ Addis offset highlight)

Squares = field blanks, circles = samples. One panel per site so each site's R vs T cloud is visible. If Warren's IMPROVE R/T figure from his 2024/2025 paper is digitised and saved at `WARREN_RT_PNG`, it's appended as a fifth panel.

In [ ]:
WARREN_RT_PNG = REPO_ROOT / 'tmp_warren_pages' / 'warren_RT.png'  # placeholder path

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()
for ax, site in zip(axes, SITE_ORDER):
    df = site_tables.get(site)
    if df is None or 'R' not in df.columns or 'T' not in df.columns:
        ax.set_visible(False); continue
    sub = df.dropna(subset=['R', 'T'])
    is_blank = sub['EC_ugm3'].isna() if 'EC_ugm3' in sub.columns else pd.Series(False, index=sub.index)
    samples = sub[~is_blank]
    blanks = sub[is_blank]
    ax.scatter(samples['T'], samples['R'], marker='o', s=30, alpha=0.7,
               color=SITE_COLORS[site], label=f'samples (n={len(samples)})')
    if not blanks.empty:
        ax.scatter(blanks['T'], blanks['R'], marker='s', s=50, alpha=0.9,
                   facecolor='none', edgecolor=SITE_COLORS[site], linewidth=1.5,
                   label=f'blanks (n={len(blanks)})')
    ax.set_xlabel('T'); ax.set_ylabel('R'); ax.set_title(SITE_LABELS[site])
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
fig.suptitle('SPARTAN HIPS R vs T per site - note Addis transmittance offset', fontsize=15)
fig.tight_layout()
save_fig(fig, 'slide10_RT_per_site')
plt.show()

if WARREN_RT_PNG.exists():
    print(f"Warren's IMPROVE R/T figure available at: {WARREN_RT_PNG}")
else:
    print(f"Warren's IMPROVE R/T figure NOT FOUND at {WARREN_RT_PNG}.\nDrop a PNG there to include it as the comparison panel.")


## Build the .pptx deck

Assembles the 11 slides per Ann's outline. Requires `python-pptx`; install with `pip install python-pptx` if missing.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)
BLANK = prs.slide_layouts[6]

def add_title(slide, text, size=28):
    box = slide.shapes.add_textbox(Inches(0.4), Inches(0.2), Inches(12.5), Inches(0.7))
    tf = box.text_frame; tf.word_wrap = True
    tf.text = text
    for p in tf.paragraphs:
        for r in p.runs: r.font.size = Pt(size); r.font.bold = True

def add_image(slide, png_path, left=0.4, top=1.0, width=12.5, height=6.2):
    if Path(png_path).exists():
        slide.shapes.add_picture(str(png_path), Inches(left), Inches(top),
                                 width=Inches(width), height=Inches(height))
    else:
        box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
        box.text_frame.text = f'[FIGURE MISSING: {png_path}]'

def add_bullets(slide, items, left=0.5, top=1.2, width=12.3, height=5.8, size=20):
    box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    tf = box.text_frame; tf.word_wrap = True
    for i, item in enumerate(items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = '• ' + item
        for r in p.runs: r.font.size = Pt(size)

# Slide 1 - intro
s = prs.slides.add_slide(BLANK)
add_title(s, 'HIPS Fabs offset at Addis Ababa - meeting with Warren White', size=30)
add_bullets(s, [
    'Goal: figure out why HIPS Fabs in Addis is anomalously high vs FTIR EC and aethalometer',
    'Four SPARTAN sites: Addis Ababa, Delhi, JPL/Pasadena, Beijing',
    'Comparison reference: post-2003 IMPROVE network (Warren et al. paper)',
    'Key question: is the offset consistent with the pixelation effect Warren describes,',
    '   or is something else going on?',
])

# Slide 2 - data and tools
s = prs.slides.add_slide(BLANK)
add_title(s, 'Data and tools')
add_bullets(s, [
    'SPARTAN filters: HIPS (Fabs, R, T, tau, t, r), FTIR EC, ChemSpec metals, ions',
    'Aethalometer (AE33): IR BCc, smoothed and raw',
    'IMPROVE (FED Wizard extract, post-2003): EC, Fabs, Fe, Volume; no R/T, no field blanks, no filter IDs',
    'Matching: filter date ± 1 day to aethalometer day_9am window',
    'MAC = 10 m²/g for HIPS → BC conversion (raw 1/Mm shown here)',
])

# Slides 3-7 - each is one figure
for slide_num, (png, title) in enumerate([
    ('slide03_fabs_vs_ec_concentration', 'Fabs (1/Mm) vs EC (µg/m³) - four sites'),
    ('slide04_fabs_vs_ftir_ec', 'Fabs (1/Mm) vs FTIR EC - four sites'),
    ('slide05_fabs_vs_aethalometer', 'Fabs (1/Mm) vs Aethalometer IR BCc'),
    ('slide06_iron', 'Iron - Fe and Fe/EC ranges'),
    ('slide07_seasonality', 'Seasonality and what we have ruled out'),
], start=3):
    s = prs.slides.add_slide(BLANK)
    add_title(s, f'{slide_num}. {title}')
    add_image(s, FIG_DIR / f'{png}.png')

# Slide 7 - append checklist as a small text box at the bottom
checklist_slide = prs.slides[6]  # the seasonality slide
box = checklist_slide.shapes.add_textbox(Inches(0.4), Inches(6.4), Inches(12.5), Inches(1.0))
tf = box.text_frame; tf.word_wrap = True
tf.text = 'Ruled out / inconclusive: ' + '; '.join(RULED_OUT_CHECKLIST)
for p in tf.paragraphs:
    for r in p.runs: r.font.size = Pt(11); r.font.italic = True

# Slide 8 - overlay
s = prs.slides.add_slide(BLANK)
add_title(s, '8. Four SPARTAN sites overlaid on IMPROVE - EC mass on filter')
add_image(s, FIG_DIR / 'slide08_overlay_mass_on_filter.png')

# Slide 9 - per-site shaded
s = prs.slides.add_slide(BLANK)
add_title(s, '9. Per site, 5-95 percentile shading on both axes')
add_image(s, FIG_DIR / 'slide09_per_site_shaded.png')

# Slide 10 - R vs T
s = prs.slides.add_slide(BLANK)
add_title(s, '10. SPARTAN R vs T per site - Addis transmittance offset')
add_image(s, FIG_DIR / 'slide10_RT_per_site.png')
if WARREN_RT_PNG.exists():
    # add separate slide for Warren's IMPROVE R/T comparison
    s = prs.slides.add_slide(BLANK)
    add_title(s, '10b. Warren et al. - IMPROVE R/T (for comparison)')
    add_image(s, WARREN_RT_PNG)

# Slide 11 - questions for Warren
s = prs.slides.add_slide(BLANK)
add_title(s, '11. Questions for Warren')
add_bullets(s, [
    'Does the Addis Fabs offset look like an extreme version of the pixelation effect, or something different?',
    'Pixelation predicts UNDER-estimation at high loadings - we see OVER-estimation. Reconcile?',
    'For SPARTAN R/T values that fall outside the IMPROVE envelope, do those filters look physically plausible to you?',
    'Do you have residual / unpublished tests on chemical-composition impacts (Fe, dust) that would be worth re-running on Addis?',
    'Is there a sensible IMPROVE subset (high-load wildfire days?) you would compare Addis against?',
])

out_pptx = OUTPUT_DIR / 'warren_meeting_slides.pptx'
prs.save(out_pptx)
print(f'Saved deck: {out_pptx}')


## Notes for Ahmad

- Run cells top-to-bottom. Figures land in `output/warren_meeting/figures/`, deck at `output/warren_meeting/warren_meeting_slides.pptx`.
- IMPROVE data: drop a pickle at `IMPROVE_DATA_PATH` with at least the columns `EC_ugm3`, `FABS_invMm`, `Volume_m3` (and optionally `Fe_ugm3`, `R`, `T`, `Date`). Until then slides 8/9/6 will show a 'PLACEHOLDER' watermark instead of the gray IMPROVE cloud.
- Warren's R/T figure: save the digitised figure from his paper as `tmp_warren_pages/warren_RT.png` to add it as a comparison panel after slide 10.
- Ann's slide 4 says 'Fabs vs FTIR EC' which is effectively the same as slide 3 with an explicit 'FTIR' axis. If you want the duplicate to be ChemSpec EC instead, swap the column in the slide-3 cell.
- Per Ann: keep raw Fabs in 1/Mm (do NOT divide by MAC = 10) so slope = MAC; this is already the case in every plot here.
- Ann emphasised making fonts and dot sizes large. The default rcParams in this notebook target font 14, scatter `s=28-36`. Bump them up if Warren needs more contrast on screen.
